# HOAN: Higher-Order Attention Network for Protein Topology
This notebook implements the HOAN architecture, focusing on lifting PDB structures into combinatorial complexes and performing contrastive learning.

## 1. Imports and Configuration
We initialize the core libraries and define the neural network components for metric learning and the HOAN architecture.

In [2]:
import os
import io
import random
import logging
import requests
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import biotite.structure as struc
import biotite.sequence as biotite_seq
import biotite.structure.io.pdb as pdb
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# GPU Optimization settings
device = torch.device("mps" if torch.mps.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.cuda.empty_cache()

print(f"Using device: {device}")

Using device: mps


In [3]:


# Define persistent storage paths within a 'deltafold' subfolder in MyDrive
BASE_DIR = './data'
os.makedirs(BASE_DIR, exist_ok=True)
PROC_DIR = os.path.join(BASE_DIR, 'hoan_processed')
os.makedirs(PROC_DIR, exist_ok=True)
RAW_DIR = os.path.join(BASE_DIR, 'hoan_raw_pdb')
os.makedirs(RAW_DIR, exist_ok=True)


print(f"Processed data will be stored in: {PROC_DIR}")
print(f"Raw data will be stored in: {RAW_DIR}")

Processed data will be stored in: ./data/hoan_processed
Raw data will be stored in: ./data/hoan_raw_pdb


## Dataset Check
We automate the retrieval of the Virome PDB dataset from Zenodo.

In [ ]:
import os
import zipfile
import subprocess

ZENODO_URL = "https://zenodo.org/record/10291581/files/Nomburg_2023_structures.zip?download=1"

def download_pdbs_fast():
    zip_path = os.path.join(RAW_DIR, "virome_pdbs.zip")

    is_valid_zip = False
    if os.path.exists(zip_path):
        try:
            with zipfile.ZipFile(zip_path, 'r') as archive:
                print("Existing zip file found. Verifying integrity...")
                is_valid_zip = True
        except zipfile.BadZipFile:
            print("Existing zip file is corrupted. Deleting and re-downloading.")
            os.remove(zip_path)
        except Exception as e:
            print(f"An unexpected error occurred while checking zip file: {e}. Deleting and re-downloading.")
            os.remove(zip_path)

    if not is_valid_zip:
        print("Downloading dataset using aria2 (multiple connections)...")
        os.makedirs(RAW_DIR, exist_ok=True)
        
        # Utilisation de subprocess.run pour un appel propre sans interférence de Shell
        cmd = [
            "aria2c", 
            "-x", "16", 
            "-s", "16", 
            "-d", RAW_DIR, 
            "-o", "virome_pdbs.zip", 
            ZENODO_URL
        ]
        
        try:
            subprocess.run(cmd, check=True)
        except subprocess.CalledProcessError as e:
            print(f"Aria2 encountered an error during download: {e}")
        except FileNotFoundError:
            print("ERROR: aria2c command not found. Ensure it is installed and in your PATH.")

        # Vérification après téléchargement
        if os.path.exists(zip_path):
            try:
                with zipfile.ZipFile(zip_path, 'r') as archive:
                    print("Download successful and zip file is valid.")
            except zipfile.BadZipFile:
                print("ERROR: Downloaded file is corrupted. Please check the URL or try again.")
                os.remove(zip_path)
        else:
            print("ERROR: Download did not complete successfully. Zip file not found.")
    else:
        print("Zip file already exists and is valid.")

download_pdbs_fast()

In [4]:
import os
import zipfile

zip_path = os.path.join(RAW_DIR, "virome_pdbs.zip")

print(f"Checking file: {zip_path}")

if os.path.exists(zip_path):
    print(f"File exists at {zip_path}")
    try:
        with zipfile.ZipFile(zip_path, 'r') as archive:
            print("File is a valid zip archive. You can proceed with processing or switch to a GPU runtime.")
            # Optional: list a few files inside to show it's readable
            print("First 5 files in archive:")
            for i, name in enumerate(archive.namelist()):
                if i >= 5: break
                print(f"- {name}")
    except zipfile.BadZipFile:
        print("ERROR: The file is present but appears to be a corrupted or incomplete zip archive. Please consider deleting it from Drive and re-running the download cell.")
    except Exception as e:
        print(f"An unexpected error occurred while verifying the zip file: {e}")
else:
    print("ERROR: The file does not exist at the specified path. Please ensure the download completed successfully.")

Checking file: ./data/hoan_raw_pdb/virome_pdbs.zip
ERROR: The file does not exist at the specified path. Please ensure the download completed successfully.


In [ ]:
import os

zip_path = os.path.join(RAW_DIR, "virome_pdbs.zip")

if os.path.exists(zip_path):
    size_bytes = os.path.getsize(zip_path)
    # Convert to human-readable format
    def convert_bytes(num):
        for unit in ['bytes', 'KB', 'MB', 'GB', 'TB']:
            if num < 1024.0:
                return f"{num:.2f} {unit}"
            num /= 1024.0

    print(f"File size of {os.path.basename(zip_path)}: {convert_bytes(size_bytes)}")
else:
    print(f"Error: The file {os.path.basename(zip_path)} does not exist at {zip_path}.")

File size of virome_pdbs.zip: 20.76 GB


In [ ]:
import re
import json

def get_plddt_from_pdb_content(content):
    """Extracts B-factors (pLDDT) from ATOM lines and returns the mean."""
    plddts = []
    for line in content.splitlines():
        if line.startswith("ATOM"):
            try:
                # B-factor is in columns 61-66
                plddt = float(line[60:66].strip())
                plddts.append(plddt)
            except (ValueError, IndexError):
                continue
    return np.mean(plddts) if plddts else 0.0

def create_stratified_subdataset(zip_path, plddt_threshold=70.0):
    print("Starting stratification process...")
    target_keywords = ['LigT', 'UL43', 'ENT4', 'I3L', 'SSB', 'phosphodiesterase', 'OB-fold']
    selected_files = []
    cluster_counts = defaultdict(int)

    with zipfile.ZipFile(zip_path, 'r') as archive:
        all_pdbs = [m for m in archive.namelist() if m.endswith('.pdb')]
        print(f"Scanning {len(all_pdbs)} structures...")

        for member_name in tqdm(all_pdbs, desc="Filtering"):
            # 1. Extract cluster info from path
            # Example path: viral_structures/family_name/protein_name.pdb
            parts = member_name.split('/')
            if len(parts) < 2: continue
            cluster_id = parts[-2]

            # Skip singletons by first pass or specific logic (here we track counts)
            # Optimization: In this dataset, singletons are often in 'undefined_family' or specific folders
            if cluster_id == 'undefined_family': continue

            try:
                with archive.open(member_name) as f:
                    content = f.read().decode('utf-8')

                # 2. Filter by pLDDT
                avg_plddt = get_plddt_from_pdb_content(content)
                if avg_plddt < plddt_threshold:
                    continue

                # 3. Check for specific target proteins (Phase 4 test cases)
                is_target = any(key.lower() in member_name.lower() for key in target_keywords)

                selected_files.append(member_name)
                cluster_counts[cluster_id] += 1

            except Exception as e:
                continue

    # 4. Final Filter: Keep only multi-member clusters (count > 1)
    final_selection = [f for f in selected_files if cluster_counts[f.split('/')[-2]] > 1]

    print(f"\nStratification Results:")
    print(f"- Initial count: {len(all_pdbs)}")
    print(f"- High-confidence multi-member count: {len(final_selection)}")

    # Save list to local storage
    subdataset_path = os.path.join(BASE_DIR, 'subdataset_files.txt')
    with open(subdataset_path, 'w') as f:
        for item in final_selection:
            f.write(f"{item}\n")

    print(f"Subdataset list saved to: {subdataset_path}")
    return final_selection

# Run the filter
zip_path = os.path.join(RAW_DIR, "virome_pdbs.zip")
#sub_files = create_stratified_subdataset(zip_path)

In [ ]:
import os
import re
import random
from collections import defaultdict
from tqdm.auto import tqdm

def extract_accession(text):
    """Extracts the accession number (e.g., YP_010085741) from a string."""
    match = re.search(r'([A-Z]{1,2}_[0-9]{5,10})', text)
    return match.group(1) if match else text

def refine_subdataset_by_clustering(cluster_tsv_path, input_list_path, stride=3):
    """
    Refined Downsampling Logic:
    1. Filters for Multi-Member Clusters only (Internal variance).
    2. Implements Cluster Striding (Even sampling of structural universe).
    3. Samples max 2 proteins per selected cluster.
    """
    if not os.path.exists(cluster_tsv_path):
        print(f"Error: {cluster_tsv_path} not found.")
        return []

    # 1. Map Accession -> Full Path
    with open(input_list_path, 'r') as f:
        available_files = [line.strip() for line in f if line.strip()]

    acc_to_path = {extract_accession(os.path.basename(p)): p for p in available_files}

    # 2. Build Cluster Map
    cluster_map = defaultdict(list)
    print("Loading and filtering clusters...")
    with open(cluster_tsv_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                rep = extract_accession(parts[0])
                member = extract_accession(parts[1])
                cluster_map[rep].append(member)

    # 3. Apply Multi-Member and Striding logic
    # Filter for clusters with > 1 member
    multi_member_reps = [rep for rep, members in cluster_map.items() if len(members) > 1]
    multi_member_reps.sort() # Ensure deterministic striding

    # Cluster Striding: select every N-th cluster
    strided_reps = multi_member_reps[::stride]

    target_keywords = ['LigT', 'UL43', 'ENT4', 'I3L', 'SSB']
    refined_selection = []
    processed_files = set()

    # Priority: Target Proteins
    print("Keeping target proteins...")
    for fn in available_files:
        if any(key.lower() in fn.lower() for key in target_keywords):
            refined_selection.append(fn)
            processed_files.add(fn)

    # Sampling: Strided Clusters
    print(f"Sampling {len(strided_reps)} strided clusters (Stride={stride})...")
    for rep_id in tqdm(strided_reps):
        members = cluster_map[rep_id]
        count = 0
        for member_id in members:
            matching_path = acc_to_path.get(member_id)
            if matching_path and matching_path not in processed_files:
                refined_selection.append(matching_path)
                processed_files.add(matching_path)
                count += 1
                if count >= 2: break

    print(f"\nDownsampling Results:")
    print(f"- Initial Clusters: {len(cluster_map)}")
    print(f"- Multi-member Clusters: {len(multi_member_reps)}")
    print(f"- Strided Clusters Selected: {len(strided_reps)}")
    print(f"- Final Protein Count: {len(refined_selection)}")

    final_path = os.path.join(BASE_DIR, 'subdataset_files_refined.txt')
    with open(final_path, 'w') as f:
        for item in refined_selection: f.write(f"{item}\n")

    global SUBDATASET_LIST_PATH
    SUBDATASET_LIST_PATH = final_path
    return refined_selection

# Run the strided cluster refinement
cluster_tsv = os.path.join(BASE_DIR, 'cluster.tsv')
input_list = os.path.join(BASE_DIR, 'subdataset_files.txt')
# Using stride=3 to reduce cluster count while maintaining coverage
#refined_files = refine_subdataset_by_clustering(cluster_tsv, input_list, stride=3)

## 3. Structural Lifting (Featurization)
In this step, we convert raw 3D coordinates into higher-order complexes. This process includes identifying Secondary Structure Elements (SSE) and building incidence matrices.

In [ ]:
import tempfile
import subprocess
from transformers import EsmTokenizer, EsmModel
from collections import defaultdict

class CCCFeaturizer:
    """
    Transforms raw atomic coordinates into a heterogeneous Combinatorial Complex (CCC).
    Correctly implements the Node-to-Edge Incidence mapping (B01).
    """
    def __init__(self, esm_model_name="facebook/esm2_t6_8M_UR50D"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = EsmTokenizer.from_pretrained(esm_model_name)
        self.esm = EsmModel.from_pretrained(esm_model_name).to(self.device)
        self.esm.eval()
        self.foldseek_alphabet = "ACDEFGHIKLMNPQRSTVWY"
        self.prof = defaultdict(float)
        self.count = 0

    def _get_foldseek_3di(self, pdb_content):
        t0 = time.time()
        with tempfile.NamedTemporaryFile(suffix=".pdb", mode='w') as tmp_pdb:
            tmp_pdb.write(pdb_content)
            tmp_pdb.flush()
            with tempfile.TemporaryDirectory() as tmp_dir:
                cmd = f"foldseek structureto3di {tmp_pdb.name} {tmp_dir}/out --threads 1"
                subprocess.run(cmd.split(), capture_output=True)
                try:
                    with open(f"{tmp_dir}/out.3di", "r") as f:
                        line = f.readline().strip().split("\t")
                        res = line[2] if len(line) >= 3 else None
                        self.prof['foldseek_call'] += time.time() - t0
                        return res
                except Exception:
                    self.prof['foldseek_call'] += time.time() - t0
                    return None

    def _lift_parsed(self, array, ca_atoms, seq, esm_emb=None, pdb_content=None):
        self.count += 1
        N = len(ca_atoms)
        if N == 0: return None

        # RANK 0: Nodes (0-cells)
        x0 = esm_emb
        v0_coords = torch.tensor(ca_atoms.coord, dtype=torch.float32, device=self.device).unsqueeze(1)

        # RANK 1: Edges (1-cells) - Vectorized
        coords = v0_coords.squeeze(1)
        dist_matrix = torch.cdist(coords, coords)
        adj_total = (dist_matrix < 8.0) & (dist_matrix > 0)
        adj_total = torch.triu(adj_total)

        edges_t = adj_total.nonzero(as_tuple=False) # [num_edges, 2]
        num_edges = edges_t.size(0)

        # Correct Incidence Matrix B01: Maps Node ID -> Edge Slot ID
        # Each edge is connected to two nodes; incidence has 2 entries per edge
        row_inc = torch.cat([edges_t[:, 0], edges_t[:, 1]])  # Node indices
        col_inc = torch.cat([torch.arange(num_edges, device=self.device),
                             torch.arange(num_edges, device=self.device)])  # Edge Slot indices
        B01_idx = torch.stack([row_inc, col_inc], dim=0)

        # RANK 2: Faces (2-cells)
        faces = []
        node_to_f_idx = np.full(N, -1, dtype=int)
        try:
            sse_raw = struc.annotate_sse(array)[:N]
            group_ids = np.cumsum(np.concatenate([[False], (sse_raw[1:] != sse_raw[:-1])]))
            for g_id in np.unique(group_ids):
                block = np.where(group_ids == g_id)[0]
                if len(block) >= 3 and sse_raw[block[0]] in ['a', 'b']:
                    f_idx = len(faces)
                    faces.append(block)
                    node_to_f_idx[block] = f_idx
        except: pass

        F = len(faces)
        B2_idx = torch.empty((2, 0), dtype=torch.long, device=self.device)
        B2_outer_idx = torch.empty((2, 0), dtype=torch.long, device=self.device)

        if F > 0 and num_edges > 0:
            u_f = node_to_f_idx[edges_t[:, 0].cpu().numpy()]
            v_f = node_to_f_idx[edges_t[:, 1].cpu().numpy()]

            # B2: Intra-SSE (Both nodes of edge in same SSE)
            mask_intra = (u_f == v_f) & (u_f != -1)
            B2_idx = torch.stack([torch.where(torch.tensor(mask_intra, device=self.device))[0],
                                  torch.tensor(u_f[mask_intra], device=self.device)], dim=0)

            # B2_outer: Inter-SSE (Edge connects two different SSEs)
            mask_inter = (u_f != v_f) & (u_f != -1) & (v_f != -1)
            if np.any(mask_inter):
                inter_edge_ids = torch.where(torch.tensor(mask_inter, device=self.device))[0]
                inter_f1 = torch.tensor(u_f[mask_inter], device=self.device)
                inter_f2 = torch.tensor(v_f[mask_inter], device=self.device)
                B2_outer_idx = torch.stack([torch.repeat_interleave(inter_edge_ids, 2),
                                            torch.stack([inter_f1, inter_f2], dim=1).flatten()], dim=0)

        x2 = torch.randn(F, 32, device=self.device) if F > 0 else torch.empty((0, 32), device=self.device)
        fs_3di_str = self._get_foldseek_3di(pdb_content) if pdb_content else None
        seq_3di = torch.tensor([self.foldseek_alphabet.find(c.upper()) for c in fs_3di_str[:N]], dtype=torch.long, device=self.device) if fs_3di_str else torch.zeros(N, dtype=torch.long, device=self.device)

        return {
            'x0': x0, 'v0': v0_coords, 'B01_idx': B01_idx, 'num_edges': num_edges,
            'x2': x2, 'B2_idx': B2_idx, 'B2_outer_idx': B2_outer_idx,
            'seq_3di': seq_3di
        }

In [ ]:
import time
import shutil
import glob
import pickle
import concurrent.futures
import torch
import re
import zipfile
from tqdm.auto import tqdm

# Configuration for local machine (MacBook Air)
DRIVE_LMDB_PATH = os.path.join(BASE_DIR, 'hoan_structures.lmdb')
PROC_DIR = os.path.join(BASE_DIR, 'hoan_processed')
os.makedirs(PROC_DIR, exist_ok=True)

# Detect CPU-only environment
device = torch.device("cpu")
print(f"Using device: {device}")

# Global featurizer instance for lifting logic only (not the model)
lite_feat = None

def lift_single_pdb_worker(args):
    """Worker function for parallel PDB parsing."""
    member_name, content, esm_emb = args
    try:
        import biotite.structure.io.pdb as pdb
        import biotite.structure as struc
        import io

        pdb_obj = pdb.PDBFile.read(io.StringIO(content))
        array = pdb.get_structure(pdb_obj, model=1)
        ca_atoms = array[(array.atom_name == 'CA') & (array.element == 'C')]

        if len(ca_atoms) < 3: return None

        # Use the global lite_feat instance which has no heavy model
        data = lite_feat._lift_parsed(array, ca_atoms, "", esm_emb=esm_emb, pdb_content=content)
        return (member_name, data)
    except Exception as e:
        return None

def reprocess_in_batches_optimized(batch_size=16, limit=None):
    """Batch lifting tuned for CPU-based machines like MacBook Air."""
    global lite_feat
    print("Initializing ESM model and tokenizer upfront...")

    # Initialize once globally
    main_featurizer = CCCFeaturizer()
    main_featurizer.esm.to(device)

    # Create a 'lite' version for the workers to use the math logic without carrying the model weights
    lite_feat = CCCFeaturizer(esm_model_name="facebook/esm2_t6_8M_UR50D")
    lite_feat.esm = None
    lite_feat.device = torch.device("cpu")

    zip_path = os.path.join(RAW_DIR, "virome_pdbs.zip")

    if not os.path.exists(SUBDATASET_LIST_PATH):
        print("Subdataset list not found.")
        return

    with open(SUBDATASET_LIST_PATH, 'r') as f:
        all_pdb_members = [line.strip() for line in f if line.strip()]

    # Check existing files to avoid redundant processing
    existing_files = set(os.listdir(PROC_DIR))
    pdb_members = []
    for name in all_pdb_members:
        out_name = name.replace('/', '_').replace('.pdb', '.pt')
        if out_name not in existing_files:
            pdb_members.append(name)

    if limit: pdb_members = pdb_members[:limit]

    print(f"Optimized Lifting: {len(pdb_members)} NEW proteins found out of {len(all_pdb_members)} total.")
    if len(pdb_members) == 0:
        print("All structures already processed.")
        return

    with zipfile.ZipFile(zip_path, 'r') as archive:
        for i in tqdm(range(0, len(pdb_members), batch_size), desc="Batched Lifting"):
            batch_names = pdb_members[i : i + batch_size]
            batch_raw = []

            for name in batch_names:
                try:
                    with archive.open(name, 'r') as f:
                        content = f.read().decode('utf-8')
                        batch_raw.append({'name': name, 'content': content})
                except: continue

            if not batch_raw: continue

            # ESM Inference on CPU (Batched)
            batch_seqs = []
            for item in batch_raw:
                try:
                    ca_lines = [l for l in item['content'].splitlines() if ' CA ' in l and l.startswith('ATOM')]
                    batch_seqs.append("A" * len(ca_lines))
                except: batch_seqs.append("A"*10)

            inputs = main_featurizer.tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
            with torch.no_grad():
                embeddings = main_featurizer.esm(**inputs).last_hidden_state

            worker_tasks = []
            for idx, item in enumerate(batch_raw):
                mask = inputs.attention_mask[idx]
                actual_len = mask.sum().item() - 2
                emb = embeddings[idx, 1:actual_len+1, :]
                worker_tasks.append((item['name'], item['content'], emb))

            # Use fewer workers on laptop to avoid swap memory slowdown
            with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
                results = list(executor.map(lift_single_pdb_worker, worker_tasks))

            for res in results:
                if res:
                    name, data = res
                    out_name = name.replace('/', '_').replace('.pdb', '.pt')
                    torch.save(data, os.path.join(PROC_DIR, out_name))

# Run the optimized loop
#reprocess_in_batches_optimized(batch_size=8)

Using device: cpu


## 3.1 Validation Sanity Check
We verify that the lifted combinatorial complexes (CCC) correctly preserve the biological information from the source PDBs.

In [ ]:
def run_sanity_check(sample_n=5):
    # Prioritize local processed directory
    target_dir_local = PROC_DIR # Use the newly defined local processed directory
    target_dir_drive = PROC_DIR

    processed_files = []

    # Check local directory first
    if os.path.exists(target_dir_local):
        for root, _, files in os.walk(target_dir_local):
            for file in files:
                if file.endswith('.pt'):
                    processed_files.append(os.path.join(root, file))

    # If local is empty, or there are no files there, check Drive
    if not processed_files and os.path.exists(target_dir_drive):
        print(f"Local processed directory {target_dir_local} is empty or does not exist. Checking Drive directory: {target_dir_drive}")
        for root, _, files in os.walk(target_dir_drive):
            for file in files:
                if file.endswith('.pt'):
                    processed_files.append(os.path.join(root, file))
    elif processed_files:
        print(f"Checking local processed directory: {target_dir_local}")
    else:
        print(f"Neither local processed directory ({target_dir_local}) nor Drive processed directory ({target_dir_drive}) contain .pt files.")
        return


    if not processed_files:
        print(f"No .pt files found in any specified processed directories.")
        return

    print(f"Checking {len(processed_files)} files...")
    samples = random.sample(processed_files, min(sample_n, len(processed_files)))

    for filename_full_path in samples:
        data = torch.load(filename_full_path) # Load directly from the full path
        print(f"File: {os.path.basename(filename_full_path)}")
        print(f"  - Nodes (x0): {data['x0'].shape[0]}")
        print(f"  - Faces (x2): {data['x2'].shape[0]}")
        print(f"  - B2 Intra-SSE: {data['B2_idx'].shape[1]}")
        print(f"  - B2 Inter-SSE (Outer): {data.get('B2_outer_idx', torch.zeros(2,0)).shape[1]}")
        print(f"  - 3Di Sequence Length: {data.get('seq_3di', torch.zeros(0)).shape[0]}")
        print("-" * 30)

run_sanity_check()

Checking local processed directory: ./data/hoan_processed
Checking 2021 files...
File: viral_structures_Phycodnaviridae_hypothetical_protein_AP053_gp111__YP_009172871__Ostreococcus_mediterraneus_virus_1__1663210.pt
  - Nodes (x0): 188
  - Faces (x2): 8
  - B2 Intra-SSE: 231
  - B2 Inter-SSE (Outer): 166
  - 3Di Sequence Length: 188
------------------------------
File: viral_structures_Coronaviridae_envelope_protein__YP_009047209__Middle_East_respiratory_syndrome-related_coronavirus__1335626.pt
  - Nodes (x0): 82
  - Faces (x2): 2
  - B2 Intra-SSE: 256
  - B2 Inter-SSE (Outer): 10
  - 3Di Sequence Length: 82
------------------------------
File: viral_structures_Herpesviridae_envelope_protein_UL43__NP_040138__Human_alphaherpesvirus_3__10335.pt
  - Nodes (x0): 406
  - Faces (x2): 16
  - B2 Intra-SSE: 1115
  - B2 Inter-SSE (Outer): 662
  - 3Di Sequence Length: 406
------------------------------
File: viral_structures_Marseilleviridae_hypothetical_protein_A3303_gp489__YP_009238994__Brazilia

In [ ]:
# Re-running sanity check now that PROC_DIR is defined and populated
run_sanity_check(sample_n=5)

Checking local processed directory: ./data/hoan_processed
Checking 2021 files...
File: viral_structures_Poxviridae_hypothetical_protein_CG743_gp165__YP_009408116__Eptesipox_virus__1329402.pt
  - Nodes (x0): 152
  - Faces (x2): 7
  - B2 Intra-SSE: 312
  - B2 Inter-SSE (Outer): 174
  - 3Di Sequence Length: 152
------------------------------
File: viral_structures_Rhabdoviridae_M_protein__YP_010086415__Maize_yellow_striate_virus__1168550.pt
  - Nodes (x0): 167
  - Faces (x2): 10
  - B2 Intra-SSE: 188
  - B2 Inter-SSE (Outer): 260
  - 3Di Sequence Length: 167
------------------------------
File: viral_structures_Baculoviridae_IAP-3__YP_009552586__Operophtera_brumata_nucleopolyhedrovirus__1046267.pt
  - Nodes (x0): 158
  - Faces (x2): 12
  - B2 Intra-SSE: 188
  - B2 Inter-SSE (Outer): 218
  - 3Di Sequence Length: 158
------------------------------
File: viral_structures_Partitiviridae_RNA-dependent_RNA_polymerase__YP_004429258__Fig_cryptic_virus__882768.pt
  - Nodes (x0): 472
  - Faces (x2)

## 4. Model Definition
We define the `HOAN` model, `TCPModule`, and `AttentionalConvolution` layers.

In [ ]:
class TCPModule(nn.Module):
    """Refined TCPModule utilizing both reduced norms and invariant projections."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        # Vd: Vector Dimension Reduction
        self.Vd = nn.Linear(3, 3, bias=False)
        # Vu: Vector Update expansion
        self.Vu = nn.Linear(in_dim + 2, 3, bias=False) # in_dim + z_norm + s_proj

        self.s_mlp = nn.Sequential(
            nn.Linear(in_dim + 2, out_dim),
            nn.ReLU(),
            nn.LayerNorm(out_dim)
        )

    def forward(self, h_s, h_v, ref_frame=None):
        if h_v.size(0) == 0:
            # Return zeros for the vector branch if empty
            return self.s_mlp(torch.cat([h_s, torch.zeros(h_s.size(0), 2, device=h_s.device)], dim=-1)), h_v

        # 1. Localized Invariant Projection
        if ref_frame is not None:
            # Project onto frame: capture orientation relative to local structure
            s_proj = torch.einsum('nci,nij->ncj', h_v, ref_frame).squeeze(1).mean(dim=-1, keepdim=True)
        else:
            # Fallback to L2 norm of the raw vector if no frame provided
            s_proj = torch.norm(h_v, dim=-1).squeeze(1).unsqueeze(-1)

        # 2. Reduced Representation for Context
        z = self.Vd(h_v.squeeze(1))
        z_norm = torch.norm(z, dim=-1, keepdim=True)

        # 3. Scalar Update Context: [h_s || ||z|| || s_proj]
        h_s_comb = torch.cat([h_s, z_norm, s_proj], dim=-1)
        h_s_out = self.s_mlp(h_s_comb)

        # 4. Vector Update: Geometric expansion from updated scalar context
        v_new = self.Vu(h_s_comb).unsqueeze(1)
        # Gated combination of original flow and new geometric orientation
        h_v_out = h_v * torch.sigmoid(z_norm).unsqueeze(-1) + v_new

        return h_s_out, h_v_out

In [ ]:
from torch_scatter import scatter_max, scatter_sum

def scattered_softmax(src, index, dim_size):
    max_val = scatter_max(src, index, dim=0, dim_size=dim_size)[0]
    score = torch.exp(src - max_val[index])
    sum_val = scatter_sum(score, index, dim=0, dim_size=dim_size)
    return score / (sum_val[index] + 1e-8)

class AttentionalConvolution(nn.Module):
    def __init__(self, src_dim, dst_dim, out_dim):
        super().__init__()
        self.attn_net = nn.Sequential(
            nn.Linear(src_dim + dst_dim, out_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(out_dim, 1)
        )
        self.tcp = TCPModule(out_dim, out_dim)

    def forward(self, s_src, v_src, s_dst, v_dst, incidence_idx):
        row, col = incidence_idx
        if row.size(0) == 0: return s_dst, v_dst

        # 1. Joint Attention
        joint_feat = torch.cat([s_src[row], s_dst[col]], dim=-1)
        weights = scattered_softmax(self.attn_net(joint_feat), col, dim_size=s_dst.size(0))

        # 2. Aggregation
        agg_s = scatter_sum(s_src[row] * weights, col, dim=0, dim_size=s_dst.size(0))
        agg_v = scatter_sum(v_src[row] * weights.unsqueeze(-1), col, dim=0, dim_size=s_dst.size(0))

        return self.tcp(s_dst + agg_s, v_dst + agg_v)

class HOAN(nn.Module):
    def __init__(self, hidden_dim=128, node_dim=1280, face_dim=32):
        super().__init__()
        self.node_embed = nn.Linear(node_dim, hidden_dim)
        self.face_embed = nn.Linear(face_dim, hidden_dim)

        self.lift_0_to_1 = AttentionalConvolution(hidden_dim, hidden_dim, hidden_dim)
        self.lift_1_to_2 = AttentionalConvolution(hidden_dim, hidden_dim, hidden_dim)
        self.sse_coadj = AttentionalConvolution(hidden_dim, hidden_dim, hidden_dim)
        self.drop_2_to_1 = AttentionalConvolution(hidden_dim, hidden_dim, hidden_dim)
        self.drop_1_to_0 = AttentionalConvolution(hidden_dim, hidden_dim, hidden_dim)

        self.proj = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))

    def forward(self, batch):
        x0, v0, x2 = batch['x0'], batch['v0'], batch['x2']
        B01_idx, B2_idx = batch['B01_idx'], batch['B2_idx']
        B2_outer = batch.get('B2_outer_idx')
        batch_idx = batch['batch_x0']
        num_edges = batch['num_edges'].sum().item()

        s0 = self.node_embed(x0)
        s2 = self.face_embed(x2) if x2.size(0) > 0 else torch.zeros(0, s0.size(1), device=s0.device)
        v2 = torch.zeros(s2.size(0), 1, 3, device=s0.device)

        # Rank-1 Destination Slot: [num_edges, hidden]
        s1 = torch.zeros(num_edges, s0.size(1), device=s0.device)
        v1 = torch.zeros(num_edges, 1, 3, device=s0.device)

        # Hierarchical Flow utilizing correct Incidence
        s1, v1 = self.lift_0_to_1(s0, v0, s1, v1, B01_idx)

        if s2.size(0) > 0:
            s2, v2 = self.lift_1_to_2(s1, v1, s2, v2, B2_idx)
            if B2_outer is not None and B2_outer.size(1) > 0:
                s2, v2 = self.sse_coadj(s1, v1, s2, v2, B2_outer)

            B2_rev = torch.stack([B2_idx[1], B2_idx[0]])
            s1, v1 = self.drop_2_to_1(s2, v2, s1, v1, B2_rev)

        B01_rev = torch.stack([B01_idx[1], B01_idx[0]])
        s0, v0 = self.drop_1_to_0(s1, v1, s0, v0, B01_rev)

        from torch_scatter import scatter_mean
        z = scatter_mean(s0, batch_idx, dim=0)
        return self.proj(z), z

In [ ]:
class NTXentLoss(nn.Module):
    """Normalized Temperature-scaled Cross Entropy Loss for contrastive learning."""
    def __init__(self, temperature=0.1):
        super(NTXentLoss, self).__init__()
        self.temperature = temperature
        self.epsilon = 1e-8

    def forward(self, z1, z2):
        """
        Calculates the contrastive loss between two batches of embeddings.
        z1, z2: [Batch, Hidden_Dim] normalized embeddings
        """
        batch_size = z1.shape[0]
        if batch_size < 2:
            return torch.tensor(0.0, device=z1.device, requires_grad=True)

        # Normalize to unit hypersphere
        z1 = F.normalize(z1, p=2, dim=1)
        z2 = F.normalize(z2, p=2, dim=1)

        # Full similarity matrix for all pairs across both views [2*B, 2*B]
        representations = torch.cat([z1, z2], dim=0)
        similarity_matrix = torch.matmul(representations, representations.T)

        # Scale by temperature
        sim_matrix_scaled = similarity_matrix / self.temperature

        # Labels for the diagonal (positive pairs are at i and i + batch_size)
        labels = torch.cat([torch.arange(batch_size) for _ in range(2)], dim=0)
        labels = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
        labels = labels.to(z1.device)

        # Mask out self-similarity (the diagonal of the matrix)
        mask = torch.eye(labels.shape[0], device=z1.device).bool()
        labels = labels[~mask].view(labels.shape[0], -1)
        sim_matrix_scaled = sim_matrix_scaled[~mask].view(sim_matrix_scaled.shape[0], -1)

        # Positive pairs are now the first column after masking for balanced batches
        # In standard NT-Xent, we maximize similarity for (i, i+B) and minimize others
        logits = sim_matrix_scaled
        # Use cross entropy to maximize the index of the positive pair
        target = torch.arange(batch_size, device=z1.device)
        target = torch.cat([target + batch_size - 1, target]) # Adjusted indices after masking

        # Since we use torch.arange approach, we can simplify labels:
        # The positive for i in [0..B-1] is at index i+B-1 in the masked row
        # The positive for i in [B..2B-1] is at index i in the masked row
        pos_idx = torch.cat([torch.arange(batch_size - 1, 2 * batch_size - 1, device=z1.device),
                            torch.arange(batch_size, device=z1.device)], dim=0)

        loss = F.cross_entropy(logits, pos_idx)
        return loss

## 5. Training and Validation
We initialize the contrastive training loop using the cluster-aware split strategy.

In [ ]:
class ViromeDataset(Dataset):
    """Dataset that samples from Foldseek clusters to ensure structural diversity."""
    def __init__(self, proc_dir, cluster_tsv=None):
        self.proc_dir = proc_dir
        all_files = {f.replace('.pt', ''): os.path.join(proc_dir, f) for f in os.listdir(proc_dir) if f.endswith('.pt')}

        if cluster_tsv and os.path.exists(cluster_tsv):
            self.clusters = defaultdict(list)
            with open(cluster_tsv, 'r') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) < 2: continue
                    rep, member = parts[0], parts[1]
                    rep_id = rep.replace('.pdb', '')
                    mem_id = member.replace('.pdb', '')
                    if mem_id in all_files:
                        self.clusters[rep_id].append(all_files[mem_id])

            self.files = [paths[0] for paths in self.clusters.values() if paths]
            print(f"Loaded {len(self.clusters)} clusters. Using {len(self.files)} representatives.")
        else:
            self.files = list(all_files.values())
            print(f"Using all {len(self.files)} files.")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        try:
            return torch.load(self.files[idx])
        except:
            return None

class HOANCollate:
    """Custom collator to pack variable-sized complexes with Translation Invariance."""
    def __call__(self, batch):
        batch = [b for b in batch if b is not None]
        if not batch: return None

        n_off, e_off, f_off = 0, 0, 0
        collated = {'x0': [], 'v0': [], 'x2': [], 'seq_3di': [],
                    'B01_idx': [], 'B2_idx': [], 'B2_outer_idx': [], 'batch_x0': [], 'num_edges': []}

        for i, data in enumerate(batch):
            n = data['x0'].size(0)
            e = data['num_edges']
            f = data['x2'].size(0)

            collated['x0'].append(data['x0'])
            # FIX: Center coordinates to ensure translation invariance
            center_of_mass = data['v0'].mean(dim=0, keepdim=True)
            collated['v0'].append(data['v0'] - center_of_mass)

            collated['x2'].append(data['x2'])
            collated['seq_3di'].append(data['seq_3di'])
            collated['batch_x0'].append(torch.full((n,), i, dtype=torch.long))
            collated['num_edges'].append(torch.tensor([e]))

            collated['B01_idx'].append(data['B01_idx'] + torch.tensor([[n_off], [e_off]]))
            if f > 0 and data['B2_idx'].size(1) > 0:
                collated['B2_idx'].append(data['B2_idx'] + torch.tensor([[e_off], [f_off]]))
            if f > 0 and data.get('B2_outer_idx') is not None and data['B2_outer_idx'].size(1) > 0:
                 collated['B2_outer_idx'].append(data['B2_outer_idx'] + torch.tensor([[e_off], [f_off]]))

            n_off += n; e_off += e; f_off += f

        return {
            'x0': torch.cat(collated['x0']).to(device),
            'v0': torch.cat(collated['v0']).to(device),
            'x2': torch.cat(collated['x2']).to(device),
            'seq_3di': torch.cat(collated['seq_3di']).to(device),
            'B01_idx': torch.cat(collated['B01_idx'], dim=1).to(device),
            'B2_idx': torch.cat(collated['B2_idx'], dim=1).to(device) if collated['B2_idx'] else torch.empty((2,0), dtype=torch.long, device=device),
            'B2_outer_idx': torch.cat(collated['B2_outer_idx'], dim=1).to(device) if collated['B2_outer_idx'] else None,
            'batch_x0': torch.cat(collated['batch_x0']).to(device),
            'num_edges': torch.cat(collated['num_edges']).to(device)
        }

In [ ]:
class ViromePairDataset(Dataset):
    def __init__(self, proc_dir, pair_list):
        self.proc_dir = proc_dir
        self.pair_list = pair_list
        if os.path.exists(proc_dir):
            self.processed_files = set(f.replace('.pt', '') for f in os.listdir(proc_dir) if f.endswith('.pt'))
        else:
            self.processed_files = set()
        print(f"[Dataset Init] {len(self.processed_files)} processed files found in {proc_dir}")

    def __len__(self):
        return len(self.pair_list)

    def __getitem__(self, idx):
        id1, id2 = self.pair_list[idx]

        if id1 not in self.processed_files or id2 not in self.processed_files:
            # Log only the first few failures to avoid spam
            if idx < 5:
                print(f"[Dataset Error] idx {idx}: One or both IDs missing from disk. {id1} in disk: {id1 in self.processed_files}, {id2} in disk: {id2 in self.processed_files}")
            return None

        try:
            path1 = os.path.join(self.proc_dir, f"{id1}.pt")
            path2 = os.path.join(self.proc_dir, f"{id2}.pt")
            return torch.load(path1), torch.load(path2)
        except Exception as e:
            print(f"[Dataset Load Error] Failed to load {id1} or {id2}: {e}")
            return None

In [ ]:
def custom_collate_pairs(batch):
    # Filter out None values returned by Dataset's __getitem__
    original_len = len(batch)
    batch = [b for b in batch if b is not None]

    if not batch:
        return None, None

    def collate_single(b_list):
        n_off, e_off, f_off = 0, 0, 0
        x0_l, v0_l, x2_l, seq_3di_l = [], [], [], []
        B01_idx_l, B2_idx_l, B2_outer_idx_l = [], [], []
        batch_x0_l, num_e_l = [], []

        for i, b in enumerate(b_list):
            # 1. The physical structure (v0) is the ground truth for node count
            N = b['v0'].size(0)
            E = b['num_edges']
            F = b['x2'].size(0)

            # --- NEW FIX: Align ESM embedding length (x0) with backbone length (v0) ---
            x0 = b['x0']
            if x0.size(0) > N:
                x0 = x0[:N, :]  # Truncate extra text-parsed atoms
            elif x0.size(0) < N:
                # Pad with zeros if text-parser missed atoms
                pad = torch.zeros(N - x0.size(0), x0.size(1), device=x0.device)
                x0 = torch.cat([x0, pad], dim=0)
            x0_l.append(x0)
            # --------------------------------------------------------------------------

            # 2. Align 3Di sequence length
            seq_3di = b['seq_3di']
            if seq_3di.size(0) > N:
                seq_3di = seq_3di[:N]
            elif seq_3di.size(0) < N:
                pad_3di = torch.zeros(N - seq_3di.size(0), dtype=torch.long, device=seq_3di.device)
                seq_3di = torch.cat([seq_3di, pad_3di], dim=0)
            seq_3di_l.append(seq_3di)

            # Center coordinates
            com = b['v0'].mean(dim=0, keepdim=True)
            v0_l.append(b['v0'] - com)

            x2_l.append(b['x2'])
            num_e_l.append(torch.tensor([E]))

            # Map Indices
            B01_idx_l.append(b['B01_idx'] + torch.tensor([[n_off], [e_off]]))
            if F > 0 and b['B2_idx'].size(1) > 0:
                B2_idx_l.append(b['B2_idx'] + torch.tensor([[e_off], [f_off]]))
            if F > 0 and b.get('B2_outer_idx') is not None and b['B2_outer_idx'].size(1) > 0:
                B2_outer_idx_l.append(b['B2_outer_idx'] + torch.tensor([[e_off], [f_off]]))

            batch_x0_l.append(torch.full((N,), i, dtype=torch.long))

            n_off += N; e_off += E; f_off += F

        return {
            'x0': torch.cat(x0_l).to(device),
            'v0': torch.cat(v0_l).to(device),
            'x2': torch.cat(x2_l).to(device),
            'B01_idx': torch.cat(B01_idx_l, dim=1).to(device) if B01_idx_l else torch.empty((2, 0), dtype=torch.long, device=device),
            'B2_idx': torch.cat(B2_idx_l, dim=1).to(device) if B2_idx_l else torch.empty((2, 0), dtype=torch.long, device=device),
            'B2_outer_idx': torch.cat(B2_outer_idx_l, dim=1).to(device) if B2_outer_idx_l else None,
            'batch_x0': torch.cat(batch_x0_l).to(device),
            'num_edges': torch.cat(num_e_l).to(device),
            'seq_3di': torch.cat(seq_3di_l).to(device)
        }

    data1_list = [b[0] for b in batch]
    data2_list = [b[1] for b in batch]
    return collate_single(data1_list), collate_single(data2_list)

In [ ]:
def get_cluster_aware_split(protein_pairs, val_split=0.2, random_seed=42):
    parent = {}
    def find(i):
        if parent.get(i, i) == i: return i
        parent[i] = find(parent[i])
        return parent[i]

    def union(i, j):
        root_i, root_j = find(i), find(j)
        if root_i != root_j: parent[root_j] = root_i

    all_proteins = set(p for pair in protein_pairs for p in pair)
    for p in all_proteins: parent[p] = p
    for p1, p2 in protein_pairs: union(p1, p2)

    clusters = defaultdict(list)
    for p in all_proteins: clusters[find(p)].append(p)

    cluster_ids = sorted(list(clusters.keys()))
    random.Random(random_seed).shuffle(cluster_ids)

    split_idx = int((1.0 - val_split) * len(cluster_ids))
    train_proteins = {p for cid in cluster_ids[:split_idx] for p in clusters[cid]}
    val_proteins = {p for cid in cluster_ids[split_idx:] for p in clusters[cid]}

    return train_proteins, val_proteins

In [ ]:
import os
import re

def extract_accession(text):
    """Extracts the unique accession number (e.g., YP_010085741) from a string."""
    match = re.search(r'([A-Z]{1,2}_[0-9]{5,10})', text)
    return match.group(1) if match else text

def mine_positive_pairs_robust(cluster_tsv_path, proc_dir):
    """
    Parses cluster.tsv and matches positive pairs to the EXACT files on disk
    by bridging them via their unique Accession IDs.
    """
    if not os.path.exists(cluster_tsv_path):
        print(f"File {cluster_tsv_path} not found.")
        return []

    print(f"Mining pairs and resolving paths via Accession IDs...")

    # 1. Catalog what is ACTUALLY on disk
    processed_files = [f for f in os.listdir(proc_dir) if f.endswith('.pt')]
    disk_map = {}
    for f in processed_files:
        acc = extract_accession(f)
        # Store the exact filename on disk (without .pt) so the Dataset loader finds it
        disk_map[acc] = f.replace('.pt', '')

    print(f"[Map] Cataloged {len(disk_map)} unique proteins currently processed on disk.")

    # 2. Parse cluster.tsv and match via Accession
    pairs = set()
    missing_count = 0

    with open(cluster_tsv_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                rep_acc = extract_accession(parts[0])
                mem_acc = extract_accession(parts[1])

                if rep_acc != mem_acc:
                    if rep_acc in disk_map and mem_acc in disk_map:
                        pairs.add((disk_map[rep_acc], disk_map[mem_acc]))
                    else:
                        missing_count += 1

    valid_pairs = list(pairs)
    print(f"[Success] Resolved {len(valid_pairs)} pairs where BOTH files exist on disk.")
    print(f"[Note] Skipped {missing_count} pairs (due to filtering or missing files).")

    return valid_pairs

In [ ]:
import random

def verify_mined_pairs_sample(pairs, sample_size=10):
    print("=== ACCURATE BIOLOGICAL REALITY CHECK ===")
    sampled = random.sample(pairs, min(sample_size, len(pairs)))

    for idx, (rep, member) in enumerate(sampled):
        print(f"\n[Pair {idx+1}]")

        # Extract the true family name from the directory path structure
        # e.g., "viral_structures/Herpesviridae/protein.pdb" -> "Herpesviridae"
        rep_dir_parts = rep.split('/')
        mem_dir_parts = member.split('/')

        # If your list has already flattened paths to strings with underscores:
        # Use a regex to catch the family name right after 'viral_structures_'
        import re
        rep_match = re.search(r'viral_structures_([A-Za-z0-9]+)_', rep)
        mem_match = re.search(r'viral_structures_([A-Za-z0-9]+)_', member)

        rep_family = rep_match.group(1) if rep_match else "Unknown"
        mem_family = mem_match.group(1) if mem_match else "Unknown"

        print(f" -> Rep Fold: {os.path.basename(rep)[:50]}...")
        print(f" -> Mem Fold: {os.path.basename(member)[:50]}...")

        if rep_family == mem_family:
            print(f" ✅ True Taxonomy: Homologous Cluster within {rep_family}")
        else:
            print(f" 🌐 True Reachability Case: Cross-Family Link! ({rep_family} <-> {mem_family})")

# Run this right before your pipeline training block
cluster_tsv_path = os.path.join(BASE_DIR, 'cluster.tsv')
pairs = mine_positive_pairs_robust(cluster_tsv_path, PROC_DIR)
verify_mined_pairs_sample(pairs, sample_size=10)

Mining pairs and resolving paths via Accession IDs...
[Map] Cataloged 2021 unique proteins currently processed on disk.
[Success] Resolved 852 pairs where BOTH files exist on disk.
[Note] Skipped 40848 pairs (due to filtering or missing files).
=== ACCURATE BIOLOGICAL REALITY CHECK ===

[Pair 1]
 -> Rep Fold: viral_structures_Phycodnaviridae_hypothetical_prot...
 -> Mem Fold: viral_structures_Phycodnaviridae_hypothetical_prot...
 ✅ True Taxonomy: Homologous Cluster within Phycodnaviridae

[Pair 2]
 -> Rep Fold: viral_structures_Phycodnaviridae_hypothetical_prot...
 -> Mem Fold: viral_structures_Phycodnaviridae_hypothetical_prot...
 ✅ True Taxonomy: Homologous Cluster within Phycodnaviridae

[Pair 3]
 -> Rep Fold: viral_structures_Poxviridae_Hypothetical_protein_S...
 -> Mem Fold: viral_structures_Poxviridae_36R_protein__NP_073421...
 ✅ True Taxonomy: Homologous Cluster within Poxviridae

[Pair 4]
 -> Rep Fold: viral_structures_Paramyxoviridae_attachment_protei...
 -> Mem Fold: viral_st

In [ ]:
def run_pairwise_training_pipeline(epochs=20, batch_size=16, val_split=0.1):
    print("--- Starting Pipeline Diagnostics ---")
    cluster_tsv_path = os.path.join(BASE_DIR, 'cluster.tsv')

    # Use the new robust accession-based miner
    valid_pairs = mine_positive_pairs_robust(cluster_tsv_path, PROC_DIR)

    if not valid_pairs:
        print("[Critical Error] No valid pairs resolved. Halting training.")
        return

    # Cluster-aware split using resolved pairs
    train_prots, val_prots = get_cluster_aware_split(valid_pairs, val_split=val_split)
    train_pairs = [p for p in valid_pairs if p[0] in train_prots]
    val_pairs = [p for p in valid_pairs if p[0] in val_prots]
    print(f"[Split] Train pairs: {len(train_pairs)}, Val pairs: {len(val_pairs)}")

    train_ds = ViromePairDataset(PROC_DIR, train_pairs)
    val_ds = ViromePairDataset(PROC_DIR, val_pairs)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_pairs, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_pairs, drop_last=True)

    model = HOAN(node_dim=320).to(device)

    checkpoint_path = os.path.join(BASE_DIR, 'hoan_best_model.pt')
    best_val_loss = float('inf')

    # --- Resume Logic ---
    if os.path.exists(checkpoint_path):
        print(f"[Resume] Loading existing checkpoint from {checkpoint_path}...")
        try:
            model.load_state_dict(torch.load(checkpoint_path, map_location=device))
            print("[Resume] Success. Resuming training from best checkpoint.")
            # Note: Since we only save the best, we don't know the exact previous loss value,
            # but the model is now initialized at its previous best performance.
        except Exception as e:
            print(f"[Resume Warning] Failed to load checkpoint: {e}. Starting from scratch.")

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = NTXentLoss(temperature=0.1).to(device)

    for epoch in range(epochs):
        # --- Training Phase ---
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')

        for b1, b2 in pbar:
            if b1 is None or b2 is None:
                continue

            optimizer.zero_grad()
            z1, _ = model(b1)
            z2, _ = model(b2)
            loss = criterion(z1, z2)
            loss.backward()
            optimizer.step()

            current_loss = loss.item()
            train_loss += current_loss
            pbar.set_postfix({'loss': f'{current_loss:.4f}', 'best_val': f'{best_val_loss:.4f}' if best_val_loss != float('inf') else 'N/A'}, refresh=True)

        avg_train_loss = train_loss / len(train_loader) if len(train_loader) > 0 else 0

        # --- Validation Phase ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for b1, b2 in tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]', leave=False):
                if b1 is None or b2 is None: continue
                z1, _ = model(b1)
                z2, _ = model(b2)
                loss = criterion(z1, z2)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader) if len(val_loader) > 0 else float('inf')

        print(f"\nEpoch {epoch+1} Summary: Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # --- Checkpointing ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), checkpoint_path)
            print(f"[Checkpoint] New best model saved to {checkpoint_path}")

# Execute updated pipeline
#run_pairwise_training_pipeline(batch_size=16)

In [ ]:
import os
import re
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from tqdm.auto import tqdm
from collections import defaultdict
from scipy.stats import ttest_ind
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, homogeneity_score
from sklearn.metrics.pairwise import cosine_similarity
from tmtools import tm_align

def run_biological_reality_check(checkpoint_name='hoan_best_model.pt'):
    print("\n=== [Phase 1] Initializing Evaluation Environment ===")
    checkpoint_path = os.path.join(BASE_DIR, checkpoint_name)
    if not os.path.exists(checkpoint_path):
        print(f"Error: Checkpoint {checkpoint_path} not found.")
        return

    # 1. Load Model
    model = HOAN(node_dim=320).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint.get('model_state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint
    model.load_state_dict(state_dict)
    model.eval()

    # 2. Extract Embeddings and Structural Coordinates
    processed_files = [os.path.join(PROC_DIR, f) for f in os.listdir(PROC_DIR) if f.endswith('.pt')]
    embeddings, sse_counts, coords_list, metadata = [], [], [], []

    print(f"Extracting latent codes for {len(processed_files)} proteins...")
    with torch.no_grad():
        for pt_file in tqdm(processed_files, desc="Inference"):
            try:
                data = torch.load(pt_file, map_location=device)
                B01_idx = data['B01_idx']
                F_cells = data['x2'].size(0)
                B2_idx = data['B2_idx'] if F_cells > 0 else torch.empty((2,0), dtype=torch.long, device=device)

                max_node_idx = B01_idx[0].max().item() if B01_idx.size(1) > 0 else -1
                safe_N = max(data['v0'].size(0), data['x0'].size(0), max_node_idx + 1)
                safe_E = data['num_edges']

                x0 = data['x0']
                if x0.size(0) < safe_N:
                    x0 = torch.cat([x0, torch.zeros(safe_N - x0.size(0), x0.size(1), device=x0.device)], dim=0)
                else: x0 = x0[:safe_N, :]

                v0 = data['v0']
                v0_centered = v0 - v0.mean(dim=0, keepdim=True)
                if v0_centered.size(0) < safe_N:
                    v0_centered = torch.cat([v0_centered, torch.zeros(safe_N - v0_centered.size(0), 1, 3, device=v0_centered.device)], dim=0)
                else: v0_centered = v0_centered[:safe_N, :]

                mini_batch = {
                    'x0': x0.to(device), 'v0': v0_centered.to(device), 'x2': data['x2'].to(device),
                    'B01_idx': B01_idx.to(device), 'B2_idx': B2_idx.to(device),
                    'B2_outer_idx': data.get('B2_outer_idx').to(device) if (F_cells > 0 and data.get('B2_outer_idx') is not None) else None,
                    'batch_x0': torch.zeros(safe_N, dtype=torch.long, device=device),
                    'num_edges': torch.tensor([safe_E], device=device)
                }

                z, _ = model(mini_batch)
                z_norm = F.normalize(z, p=2, dim=1).cpu().numpy().flatten()

                embeddings.append(z_norm)
                sse_counts.append(F_cells)
                coords_list.append(data['v0'].squeeze(1).cpu().numpy())
                filename = os.path.basename(pt_file)
                metadata.append({
                    'id': filename.replace('.pt',''),
                    'family': re.search(r'viral_structures_([A-Za-z0-9]+)_', filename).group(1) if 'viral_structures_' in filename else "Unknown"
                })
            except: continue

    X = np.array(embeddings)
    df_meta = pd.DataFrame(metadata)

    print("\n=== [Analysis 3] Taxonomic Silhouette & Structural ground-truth ===")
    n_clusters = min(df_meta['family'].nunique(), 30)
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    clusters = kmeans.fit_predict(X)
    df_meta['cluster'] = clusters

    print(f"-> Silhouette Score: {silhouette_score(X, clusters):.4f}")
    print(f"-> Homogeneity Score: {homogeneity_score(df_meta['family'], clusters):.4f}")

    # TM-Align Validation
    print("\nRunning TM-Align sample analysis (Intra vs Inter cluster)...")
    intra_tm, inter_tm = [], []
    indices = np.arange(len(df_meta))

    # Sample 50 pairs for each
    for _ in range(50):
        # Intra-cluster pair
        c = np.random.choice(df_meta['cluster'].unique())
        c_idxs = indices[df_meta['cluster'] == c]
        if len(c_idxs) > 1:
            i, j = np.random.choice(c_idxs, 2, replace=False)
            try:
                res = tm_align(coords_list[i], coords_list[j], "A"*len(coords_list[i]), "A"*len(coords_list[j]))
                intra_tm.append(res.tm_norm_chain1)
            except: pass

        # Inter-cluster pair
        c1, c2 = np.random.choice(df_meta['cluster'].unique(), 2, replace=False)
        i = np.random.choice(indices[df_meta['cluster'] == c1])
        j = np.random.choice(indices[df_meta['cluster'] == c2])
        try:
            res = tm_align(coords_list[i], coords_list[j], "A"*len(coords_list[i]), "A"*len(coords_list[j]))
            inter_tm.append(res.tm_norm_chain1)
        except: pass

    print(f"-> Mean Intra-Cluster TM-score: {np.mean(intra_tm):.4f}")
    print(f"-> Mean Inter-Cluster TM-score: {np.mean(inter_tm):.4f}")
    print(f"-> TM Enrichment Ratio: {np.mean(intra_tm)/np.mean(inter_tm):.2f}x")

if __name__ == '__main__':
    run_biological_reality_check()


=== [Phase 1] Initializing Evaluation Environment ===
Extracting latent codes for 2021 proteins with Safety Padding...


Inference: 100%|██████████| 2021/2021 [02:47<00:00, 12.10it/s]



=== [Analysis 1] Secondary Structure Layout Differentiation ===
-> Layout Cosine Similarity: 0.2700
-> Statistically Separated Dimensions: 114/128

=== [Analysis 2] Homology Recognition Check ===
Mining pairs and resolving paths via Accession IDs...
[Map] Cataloged 2021 unique proteins currently processed on disk.
[Success] Resolved 852 pairs where BOTH files exist on disk.
[Note] Skipped 40848 pairs (due to filtering or missing files).
-> Intra-Homolog vs Inter-Random Distance Ratio: 0.2555

=== [Analysis 3] Taxonomic Silhouette & Homogeneity ===


/Users/liamderay/Documents/PASCAL/deltafold/.venv/lib/python3.11/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


-> Silhouette Score: 0.2247
-> Homogeneity Score: 0.2015

--- Biological Purity Breakdown (Top 5 Clusters) ---
Cluster 0: Size=87 | Primary: Phycodnaviridae (36.8% purity)
Cluster 1: Size=59 | Primary: Baculoviridae (15.3% purity)
Cluster 2: Size=90 | Primary: Herpesviridae (20.0% purity)
Cluster 3: Size=59 | Primary: Herpesviridae (25.4% purity)
Cluster 4: Size=36 | Primary: Baculoviridae (33.3% purity)


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from tmtools import tm_align
from collections import defaultdict
import torch.nn.functional as F
import re

def extract_accession(text):
    """Extracts the unique accession number (e.g., YP_010085741) from a string."""
    match = re.search(r'([A-Z]{1,2}_[0-9]{5,10})', text)
    return match.group(1) if match else text

def analyze_structural_consistency(cluster_tsv_path, proc_dir, max_clusters=3, max_prots_per_cluster=5):
    print(f"--- Structural Consistency Analysis (Top {max_clusters} Clusters) ---")

    # 1. Map local Accession -> Full Filename on disk
    existing_files = {}
    if not os.path.exists(proc_dir):
        print(f"Error: {proc_dir} does not exist.")
        return pd.DataFrame()

    for f in os.listdir(proc_dir):
        if f.endswith('.pt'):
            acc = extract_accession(f)
            existing_files[acc] = f

    print(f"Found {len(existing_files)} processed proteins in {proc_dir}")

    # 2. Parse cluster.tsv and build cluster_map using accessions
    cluster_map = defaultdict(list)
    with open(cluster_tsv_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 2: continue

            rep_acc = extract_accession(parts[0])
            mem_acc = extract_accession(parts[1])

            # We only track members we actually have processed on disk
            if mem_acc in existing_files:
                if mem_acc not in cluster_map[rep_acc]:
                    cluster_map[rep_acc].append(mem_acc)

    # 3. Filter for multi-member clusters (where we have >=2 files)
    valid_reps = [r for r, m in cluster_map.items() if len(m) >= 2]
    print(f"Identified {len(valid_reps)} multi-member clusters matching local files.")
    selected_reps = valid_reps[:max_clusters]

    if not selected_reps:
        print("No matching clusters found. Ensure cluster.tsv accessions match PROC_DIR file accessions.")
        return pd.DataFrame()

    # 4. Load Model
    model = HOAN(node_dim=320).to(device)
    checkpoint_path = os.path.join(BASE_DIR, 'hoan_best_model.pt')
    if not os.path.exists(checkpoint_path):
        print("Error: Model checkpoint not found.")
        return pd.DataFrame()

    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    results = []
    for rep_acc in selected_reps:
        members = cluster_map[rep_acc][:max_prots_per_cluster]
        print(f"\nAnalyzing Cluster Grouped by: {rep_acc} ({len(members)} proteins)")

        latent_vectors, coords_list = [], []

        for mem_acc in members:
            data = torch.load(os.path.join(proc_dir, existing_files[mem_acc]), map_location=device)
            # Center coords for consistency
            v0_centered = data['v0'] - data['v0'].mean(dim=0, keepdim=True)

            mb = {
                'x0': data['x0'].to(device), 'v0': v0_centered.to(device), 'x2': data['x2'].to(device),
                'B01_idx': data['B01_idx'].to(device), 'B2_idx': data['B2_idx'].to(device),
                'batch_x0': torch.zeros(data['x0'].size(0), dtype=torch.long, device=device),
                'num_edges': torch.tensor([data['num_edges']], device=device)
            }

            with torch.no_grad():
                z, _ = model(mb)
                latent_vectors.append(F.normalize(z, p=2, dim=1).cpu().numpy())
                coords_list.append(data['v0'].squeeze(1).cpu().numpy())

        # 5. Metrics
        latents = np.vstack(latent_vectors)
        cos_sims = cosine_similarity(latents)
        mask = np.triu(np.ones(cos_sims.shape), k=1).astype(bool)
        mean_cos = cos_sims[mask].mean() if mask.any() else 1.0

        tm_scores = []
        for i in range(len(coords_list)):
            for j in range(i + 1, len(coords_list)):
                try:
                    res = tm_align(coords_list[i], coords_list[j], "A"*len(coords_list[i]), "A"*len(coords_list[j]))
                    tm_scores.append(res.tm_norm_chain1)
                except: pass

        mean_tm = np.mean(tm_scores) if tm_scores else 0.0
        print(f" -> Mean Latent Cosine: {mean_cos:.4f} | Mean TM-score: {mean_tm:.4f}")
        results.append({'cluster_rep': rep_acc, 'cos': mean_cos, 'tm': mean_tm})

    return pd.DataFrame(results)

# Execution
cluster_tsv = os.path.join(BASE_DIR, 'cluster.tsv')
if os.path.exists(cluster_tsv):
    df_res = analyze_structural_consistency(cluster_tsv, PROC_DIR)
    if not df_res.empty: display(df_res)
else:
    print("cluster.tsv missing.")

In [ ]:
import os
import urllib.request
import json
import torch
import torch.nn.functional as F
import numpy as np
import biotite.structure.io.pdb as pdb_io
import io
import time
from tqdm.auto import tqdm
from transformers import EsmTokenizer, EsmModel

def discover_viral_mimic(uniprot_id="Q96PZ0", host_name="Human ENT4"):
    """
    Downloads a Host Eukaryotic structure from AlphaFold DB using the API,
    processes it through HOAN using real ESM-2 embeddings, and searches the viral latent space.
    """
    print(f"\n=== Phase 4: Cross-Kingdom Mimicry Discovery ===")

    # 1. Download Host Structure from AlphaFold via API Resolution
    host_pdb_path = os.path.join(RAW_DIR, f"{host_name.replace(' ', '_')}.pdb")

    if not os.path.exists(host_pdb_path):
        print(f"Querying AlphaFold DB API for {host_name} ({uniprot_id})...")
        try:
            api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
            req = urllib.request.Request(api_url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as response:
                api_data = json.loads(response.read().decode())

            if api_data and len(api_data) > 0:
                af_url = api_data[0]['pdbUrl']
                print(f"Resolved URL: {af_url}")

                req_pdb = urllib.request.Request(af_url, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req_pdb) as stream_in, open(host_pdb_path, 'wb') as stream_out:
                    stream_out.write(stream_in.read())
                print(f"Successfully downloaded {host_name} structure.")
            else:
                raise ValueError("Empty API response payload.")
        except Exception as e:
            print(f"\u274c AlphaFold API resolution failed: {e}. Trying fallback...")
            try:
                fallback_url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb"
                req_fb = urllib.request.Request(fallback_url, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req_fb) as stream_in, open(host_pdb_path, 'wb') as stream_out:
                    stream_out.write(stream_in.read())
            except Exception as e2:
                print(f"Critical Download Failure: {e2}.")
                return

    # 2. Setup Featurizer and ESM-2 for proper x0 embeddings
    esm_model_name = "facebook/esm2_t6_8M_UR50D"
    print(f"Loading ESM-2 ({esm_model_name}) for host featurization...")
    tokenizer = EsmTokenizer.from_pretrained(esm_model_name)
    esm_model = EsmModel.from_pretrained(esm_model_name).to(device)
    esm_model.eval()

    local_featurizer = CCCFeaturizer(esm_model_name=esm_model_name)
    local_featurizer.esm = esm_model

    # 3. Lift the Host Structure
    print(f"Lifting {host_name} into a Combinatorial Complex...")
    with open(host_pdb_path, 'r') as f:
        content = f.read()

    pdb_obj = pdb_io.PDBFile.read(io.StringIO(content))
    array = pdb_io.get_structure(pdb_obj, model=1)
    ca_atoms = array[(array.atom_name == 'CA') & (array.element == 'C')]

    # Extract sequence for ESM
    seq = "".join([biotite_seq.ProteinSequence.convert_letter_3to1(r) for r in ca_atoms.res_name])

    inputs = tokenizer(seq, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        embeddings = esm_model(**inputs).last_hidden_state
        # Remove [CLS] and [SEP] tokens
        host_esm_emb = embeddings[0, 1:-1, :]

    host_data = local_featurizer._lift_parsed(array, ca_atoms, seq, esm_emb=host_esm_emb, pdb_content=content)

    # 4. Load Trained HOAN Model and Project Host
    print("Projecting Host into the Viral Latent Space...")
    model = HOAN(node_dim=320).to(device)
    checkpoint_path = os.path.join(BASE_DIR, 'hoan_best_model.pt')
    if not os.path.exists(checkpoint_path):
        print("Error: hoan_best_model.pt not found.")
        return

    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    with torch.no_grad():
        v0 = host_data['v0'] - host_data['v0'].mean(dim=0, keepdim=True)
        mini_batch = {
            'x0': host_data['x0'].to(device), 'v0': v0.to(device), 'x2': host_data['x2'].to(device),
            'B01_idx': host_data['B01_idx'].to(device), 'B2_idx': host_data['B2_idx'].to(device),
            'B2_outer_idx': host_data.get('B2_outer_idx'),
            'num_edges': torch.tensor([host_data['num_edges']], device=device),
            'batch_x0': torch.zeros(host_data['x0'].size(0), dtype=torch.long, device=device)
        }
        _, z_host = model(mini_batch)
        z_host_norm = F.normalize(z_host, p=2, dim=1).cpu().numpy().flatten()

    # 5. Search the Viral Database
    processed_files = [f for f in os.listdir(PROC_DIR) if f.endswith('.pt')]
    print(f"Scanning {len(processed_files)} viral proteins for structural mimics...")

    best_sim = -1
    best_candidate = None

    for fname in tqdm(processed_files, desc="Scanning"):
        try:
            data = torch.load(os.path.join(PROC_DIR, fname), map_location=device)
            v0_v = data['v0'] - data['v0'].mean(dim=0, keepdim=True)
            N_v = v0_v.size(0)
            x0_v = data['x0']
            if x0_v.size(0) > N_v: x0_v = x0_v[:N_v]
            elif x0_v.size(0) < N_v: x0_v = torch.cat([x0_v, torch.zeros(N_v - x0_v.size(0), x0_v.size(1), device=x0_v.device)])

            mb_v = {
                'x0': x0_v.to(device), 'v0': v0_v.to(device), 'x2': data['x2'].to(device),
                'B01_idx': data['B01_idx'].to(device), 'B2_idx': data['B2_idx'].to(device),
                'B2_outer_idx': data.get('B2_outer_idx'),
                'num_edges': torch.tensor([data['num_edges']], device=device),
                'batch_x0': torch.zeros(N_v, dtype=torch.long, device=device)
            }
            with torch.no_grad():
                _, z_v = model(mb_v)
                z_v_norm = F.normalize(z_v, p=2, dim=1).cpu().numpy().flatten()

                sim = np.dot(z_host_norm, z_v_norm)
                if sim > best_sim:
                    best_sim = sim
                    best_candidate = fname
        except: continue

    print(f"\n=== Discovery Results ===")
    print(f"Top Viral Mimic Candidate for {host_name}:")
    print(f"-> File: {best_candidate}")
    print(f"-> Structural Similarity Score: {best_sim:.4f}")
    if best_candidate and "hypothetical" in best_candidate.lower():
        print("\ud83d\udca1 NOTE: This viral protein is unannotated ('hypothetical'). Function assigned via mimicry!")

discover_viral_mimic(uniprot_id="Q7RTT9", host_name="Human ENT4")


=== Phase 4: Cross-Kingdom Mimicry Discovery ===


/Users/liamderay/Documents/PASCAL/deltafold/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Lifting Human ENT4 into a Combinatorial Complex...
Projecting Host into the Viral Latent Space...
Scanning 2021 viral proteins for structural mimics...


Scanning: 100%|██████████| 2021/2021 [02:42<00:00, 12.43it/s]


=== Discovery Results ===
Top Viral Mimic Candidate for Human ENT4:
-> File: viral_structures_Baculoviridae_baculovirus_repeated_ORF__YP_009316173__Anticarsia_gemmatalis_multiple_nucleopolyhedrovirus__268591.pt
-> Structural Similarity Score: 0.9715
